In this notebook there is an example how the models were trained to predict ticket submissions.

First run the imports and read all the relevant files required for this notebook

In the "Create Dataset" section, the sub-section "Auxiliary Files" should always be run, irrespective of whether the dataset is built from scratch or loaded from disk.
    To make loading times in between seesion faster, the postive/negative datasets were saved. If present in the Database the respective variables can be loaded instead of created by scratch.

Model training is divided into multi- and binary-classification. Inside each task the model training follows a consistent pattern:
    - First optuna is used to optimize hyperparameters
    - From the models outputted by last step it is tested:
        - F1 score on the test set
        - Feature importance

For the Binary-classification task, an additional PU learning algorithm is used to obtain reliable negatives from those mined previously. 

# Imports

In [ ]:
%load_ext autoreload
%autoreload 2
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import pandas as pd
import json
import sys
import os
import ssl

sys.path.append("../src")

from sentence_transformers import SentenceTransformer

import matplotlib.pyplot as plot
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
import matplotlib.dates as mdates
import matplotlib.gridspec as gridspec

from sklearn.feature_selection import SelectKBest, mutual_info_classif
import plotly.io as pio
import plotly.express as px
from joblib import Parallel, delayed

from functools import partial
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
import pandas as pd
import numpy as np
import scipy
import sklearn as sk
import ruptures as rpt
import pymannkendall as mk
from collections import Counter, defaultdict
from tqdm import tqdm
import statsmodels.api as sm
from itertools import combinations
from bs4 import BeautifulSoup
import re
from sklearn.preprocessing import MultiLabelBinarizer
from scipy.sparse import csr_matrix
from mlxtend.frequent_patterns import apriori, association_rules
from prefixspan import PrefixSpan
from functools import reduce
from statsmodels.stats.proportion import proportions_ztest
from scipy.stats import ttest_ind
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, AutoPeftModelForCausalLM, PeftModel

import torch
import torch.nn as nn
import torch.nn.functional as F
import copy

from bisect import bisect_left,bisect_right
from sqlalchemy import create_engine
import dask.dataframe as dd
import duckdb
import subprocess
import pickle
import tensorflow_hub as hub
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from nltk.tokenize import word_tokenize
import nltk
from bertopic import BERTopic
# import warnings
# warnings.filterwarnings("ignore")
# warnings.filterwarnings("default")

from sklearn.metrics import (
silhouette_score,
davies_bouldin_score,
calinski_harabasz_score
)
from sklearn.cluster import AgglomerativeClustering
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, fcluster

import shap
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


from datasets import Dataset

from global_utils.files_utils import create_tree_folder
from global_utils.graphs_utils import get_subplots, prepare_subplots, fancy_histogram, color_dic, arrange_twin_plots
from model_training.model_utils import class_short_names, new_class_short_names, preprocess_events_counts, get_event_count_no_tot, get_event_count, model_train, get_scores_gaps, get_model_stats



# Read Files

## Files

In [ ]:
DATABASE_DIR = "../Database"
TICKET_DIR = os.path.join(DATABASE_DIR, "Ticket_Extraction")
EVENT_DIR = os.path.join(DATABASE_DIR, "Event_Download")

fontsize = 18
if os.path.exists(DATABASE_DIR):
    print("Can see Database_Dir")

if os.path.exists(TICKET_DIR):
    print("Can see Ticket_Dir")


In [ ]:
charger_locations = pd.read_parquet(os.path.join(EVENT_DIR, "category_bucket_map.parquet"))
all_names = pd.read_parquet(os.path.join(EVENT_DIR, "all_names.parquet"))["name"].dropna().tolist()
min_max_events = pd.read_csv(os.path.join(DATABASE_DIR, "min_max_events.csv"))
sicharge_family = pd.read_csv(os.path.join(DATABASE_DIR, "SichargeD_Family_min_max.csv"))

In [ ]:
new_to_old = {
    key_new : [key for key, val in class_short_names.items() if val == new_class_short_names[key_new]][0]
    for key_new, val_new in new_class_short_names.items()
}

# Create Dataset

## Auxiliary Files

In [ ]:

summaries = []
names = []
charger_id = []
all_tickets   = os.path.join(TICKET_DIR, "Tickets_Trimmed_Summary")
for index, file_name in tqdm(enumerate(os.listdir(all_tickets))):
    names.append(file_name)
    with open(os.path.join(all_tickets, file_name), "r") as in_file:
        summaries.append(in_file.read())
    json_load = json.load(open(os.path.join(TICKET_DIR, "Tickets_Trimmed", file_name.split(".")[0]+".json"), "r"))
    charger_id.append([json_load["chargerID"], json_load["created_on"], json_load["incident_id"], int(index)])
    

charger_id = np.array(charger_id)

In [ ]:

model_name = "allenai-specter"
embeddings = np.load(os.path.join(TICKET_DIR, "Ticket_Embeddings", f"{model_name}_embeddings.npy"))
k_means_labels = np.load(os.path.join(TICKET_DIR, "Ticket_Embeddings", f"{model_name}_k_means_labels.npy"))

In [ ]:

specific_tickets = charger_id
per_charger = defaultdict(list)
for el in specific_tickets:
    per_charger[el[0]].append([pd.Timestamp(str(el[1])), el[2], int(el[3])])

In [ ]:
# all_names_used = all_names.copy()
# if use_regex:
#     all_names_used = [re.sub(r'\d+', 'X', name) for name in all_names_used]
#     all_names_used = sorted(list(set(all_names_used)))

# name_idx = {n: i for i, n in enumerate(all_names_used)}

In [ ]:
name_idx = np.load(os.path.join(TICKET_DIR, "name_idx.npy"), allow_pickle=True).item()
all_names_used = sorted(name_idx.keys())

## Create

In [ ]:
use_labels = k_means_labels
use_regex=True
threshold_gap = pd.Timedelta("1W")
cols = ["name"]

### Positive

In [ ]:
first_filter_index = []
for val, group in charger_locations.groupby("bucket"):
    chargers_in_bucket = group["device_id"].tolist()

    for charger in tqdm(chargers_in_bucket):
        min_event = sicharge_family[sicharge_family["charging_station_id"] == charger]["min"].values[0]
        if pd.isna(min_event):
            continue
        min_event = pd.to_datetime(min_event).tz_localize('UTC')
        for date,incident_id,index in per_charger[charger]:
            if date < min_event:
                continue


            first_filter_index.append(index)

In [ ]:
positive = []
y_positive = []
for val, group in charger_locations.groupby("bucket"):
    chargers_in_bucket = group["device_id"].tolist()
    
    events = pd.read_parquet(os.path.join(EVENT_DIR, "Buckets", f"bucket_{val}.parquet")).copy()
    events = preprocess_events_counts(events)
    
    events["event_time"] = (events["event_time"]).dt.tz_localize("UTC")

    for charger in tqdm(chargers_in_bucket):
        curr_charger = None

        for date,incident,index in per_charger[charger]:
            if not index in first_filter_index:
                continue
            else:
                if curr_charger is None:
                    specific_events = events[events["device_id"] == charger]#.copy()
                    min_event = specific_events["event_time"].min()
                    curr_charger = charger
                    total_counts = specific_events.groupby(cols, observed=True, as_index=False, dropna=False).agg(count_total=("event_time", "size")).reset_index(drop=1)
            # if date < min_event:
                # continue

            temp = get_event_count(specific_events, date, threshold_gap, total_counts, threshold_event=0, cols=cols)

            if len(temp) > 0:
                positive.append(temp)
                y_positive.append(k_means_labels[index])


In [ ]:
X_positive = np.zeros((len(positive), len(all_names_used)))
X_total_positive = np.zeros((len(positive), len(all_names_used)))
for i, arr in enumerate(positive):
    for name, cnt, total in arr:
        idx = name_idx[name]
        X_positive[i, idx] = cnt
        X_total_positive[i, idx] = total

In [ ]:
# np.save(os.path.join(EVENT_DIR, "X_positive_2M.npy"), X_positive)
# np.save(os.path.join(EVENT_DIR, "X_total_positive_2M.npy"), X_total_positive)
# np.save(os.path.join(EVENT_DIR, "y_positive_2M.npy"), y_positive)


### Negative

In [ ]:
guard_threshold = pd.Timedelta("2W")
period_threshold = pd.Timedelta("4W")

max_iter = 5000
total_periods = []
np.random.seed(42)
negative_candidates = []

for charger, initial_event_time, final_event_time in tqdm(sicharge_family[["charging_station_id", "min", "max"]].values):
    final_event_time = pd.to_datetime(final_event_time).tz_localize("UTC")
    initial_event_time = pd.to_datetime(initial_event_time).tz_localize("UTC")
    if pd.isna(initial_event_time) or pd.isna(final_event_time):
        continue
    number_of_days = (final_event_time.normalize() - initial_event_time.normalize()).total_seconds()/(3600*24)
    if number_of_days < 10:
       continue
    specific_tickets = per_charger[charger]
    dates = [el[0] for el in specific_tickets]
    samples = number_of_days // 30
    index = 0
    current_negatives = []
    while index < max_iter and samples > 0:
        index += 1
        candidate_date = initial_event_time + pd.Timedelta(days=np.random.uniform(7, number_of_days-7))
        if any ([dt - period_threshold <= candidate_date <= dt + period_threshold for dt in dates] ) or any([curr[1] - guard_threshold <= candidate_date <= curr[1] + guard_threshold for curr in current_negatives]):
            continue
        current_negatives.append((charger, candidate_date))
        samples -= 1
    negative_candidates.extend(current_negatives)


In [ ]:
per_charger_negative = defaultdict(list)
for el in negative_candidates:
    per_charger_negative[el[0]].append(el[1])

chargers_in_negatives = list(per_charger_negative.keys())


In [ ]:
negative = []

for val, group in charger_locations.groupby("bucket"):
    chargers_in_bucket = group["device_id"].tolist()

    events = pd.read_parquet(os.path.join(EVENT_DIR, "Buckets", f"bucket_{val}.parquet"))
    events = preprocess_events_counts(events)

    events["event_time"] = (events["event_time"]).dt.tz_localize("UTC")

    for charger in tqdm(chargers_in_bucket):
        if not charger in chargers_in_negatives:
            continue
    
        curr_charger = None
        for date in per_charger_negative[charger]:
            if curr_charger is None:
                specific_events = events[events["device_id"] == charger]#.copy()
                curr_charger = charger
                min_event = specific_events["event_time"].min()
                total_counts = specific_events.groupby(cols, observed=True, as_index=False, dropna=False).agg(count_total=("event_time", "size")).reset_index(drop=1)
            
            temp = get_event_count(specific_events, date, threshold_gap, total_counts, threshold_event=0, cols=cols)
            if len(temp) > 0:
                negative.append(temp)

In [ ]:
X_negative = np.zeros((len(negative), len(all_names_used)))
X_total_negative = np.zeros((len(negative), len(all_names_used)))
for i, neg in tqdm(enumerate(negative), total=len(negative)):
    for name, count, total in neg:
        if name in name_idx:
            X_negative[i, name_idx[name]] = count
            X_total_negative[i, name_idx[name]] = total


In [ ]:
# np.save(os.path.join(EVENT_DIR, "X_negative.npy"), X_negative)
# np.save(os.path.join(EVENT_DIR, "X_total_negative.npy"), X_total_negative)

## Load

In [ ]:
X_positive = np.load(os.path.join(EVENT_DIR, "X_positive.npy"))
X_total_positive = np.load(os.path.join(EVENT_DIR, "X_total_positive.npy"))
y_positive = np.load(os.path.join(EVENT_DIR, "y_positive.npy"))
X_negative = np.load(os.path.join(EVENT_DIR, "X_negative.npy"))
X_total_negative = np.load(os.path.join(EVENT_DIR, "X_total_negative.npy"))
y_negative= np.full(len(X_negative), -1)  

## Normalize

In [ ]:
tfdif = sk.feature_extraction.text.TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
tfdif.fit(X_total_positive)
X_positive_norm = tfdif.transform(X_positive).toarray()


In [ ]:
np.random.seed(42)
# indexes = np.random.choice(X_negative.shape[0], size=X_positive.shape[0]*2, replace=False)
indexes = np.arange(X_negative.shape[0])

X_full = np.concatenate([X_positive, X_negative[indexes]], axis=0)
X_total_full = np.concatenate([X_total_positive, X_total_negative[indexes]], axis=0)
y_full = np.concatenate([y_positive, y_negative[indexes]], axis=0)
y_binary = np.array([1 if y != -1 else 0 for y in y_full])

In [ ]:
tfdif_full = sk.feature_extraction.text.TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)
tfdif_full.fit(X_total_full)
X_full_norm = tfdif_full.transform(X_full).toarray()

# Model Train

In [ ]:
def objective_function(trial,x,y, average="macro"):
    
    # Choose classifier type
    classifier_name = trial.suggest_categorical('classifier', ['RandomForest', 'SVM', 'LogisticRegression', 'AdaBoost', 'GradientBoosting', 'KNN'])
    # classifier_name = trial.suggest_categorical('classifier', ['RandomForest', 'SVM', 'LogisticRegression'])
    # classifier_name = "RandomForest"
    # classifier_name = "GradientBoosting"
    # classifier_name = "LogisticRegression"

    # Build model based on classifier choice
    if classifier_name == 'RandomForest':
        n_estimators = trial.suggest_int('n_estimators', 10, 200)
        max_depth = trial.suggest_int('max_depth', 1, 20)
        class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
        model = sk.ensemble.RandomForestClassifier(
            n_estimators=n_estimators, 
            max_depth=max_depth, 
            class_weight=class_weight,
            random_state=42
        )
    
    elif classifier_name == 'SVM':
        C = trial.suggest_float('svm_C', 0.1, 10.0, log=True)
        kernel = trial.suggest_categorical('svm_kernel', ['linear', 'rbf', 'poly'])
        class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
        model = sk.svm.SVC(
            C=C, 
            kernel=kernel,
            class_weight=class_weight,
            random_state=42
        )
    
    elif classifier_name == 'LogisticRegression':
        C = trial.suggest_float('lr_C', 0.01, 10.0, log=True)
        solver = trial.suggest_categorical('lr_solver', ['lbfgs', 'saga', 'newton-cg'])
        class_weight = trial.suggest_categorical('class_weight', ['balanced', None])
        model = sk.linear_model.LogisticRegression(
            C=C, 
            solver=solver,
            class_weight=class_weight,
            max_iter=1000, 
            random_state=42
        )
    
    elif classifier_name == 'AdaBoost':
        n_estimators = trial.suggest_int('ada_n_estimators', 10, 200)
        learning_rate = trial.suggest_float('ada_learning_rate', 0.01, 2.0, log=True)
        max_depth = trial.suggest_int('ada_max_depth', 1, 10)
        weak_learner = sk.tree.DecisionTreeClassifier(max_depth=max_depth, random_state=42)
        model = sk.ensemble.AdaBoostClassifier(
            estimator=weak_learner,
            n_estimators=n_estimators,
            learning_rate=learning_rate,
            random_state=42
        )
    
    elif classifier_name == 'GradientBoosting':
        n_estimators = trial.suggest_int('gb_n_estimators', 10, 200)
        max_depth = trial.suggest_int('gb_max_depth', 1, 15)
        learning_rate = trial.suggest_float('gb_learning_rate', 0.01, 0.5, log=True)
        subsample = trial.suggest_float('gb_subsample', 0.5, 1.0)
        model = sk.ensemble.GradientBoostingClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            learning_rate=learning_rate,
            subsample=subsample,
            random_state=42
        )
    
    elif classifier_name == 'KNN':
        n_neighbors = trial.suggest_int('knn_n_neighbors', 1, 30)
        weights = trial.suggest_categorical('knn_weights', ['uniform', 'distance'])
        metric = trial.suggest_categorical('knn_metric', ['euclidean', 'manhattan', 'minkowski'])
        model = sk.neighbors.KNeighborsClassifier(
            n_neighbors=n_neighbors,
            weights=weights,
            metric=metric
        )
    
    # Evaluate model with K-fold cross-validation
    test_scores = []
    gap = []
    Kfold = sk.model_selection.KFold(n_splits=5, random_state=42, shuffle=True)
    

    for train, test in Kfold.split(x, y):
        model.fit(x[train], y[train])
        
        # Test score
        y_pred = model.predict(x[test])
        test_score = sk.metrics.f1_score(y[test], y_pred, average=average)
        test_scores.append(test_score)
        
        # Train score (for gap calculation)
        y_pred = model.predict(x[train])
        train_score = sk.metrics.f1_score(y[train], y_pred, average=average)
        gap.append(train_score - test_score)
    
    mean_test_score = np.mean(test_scores)
    mean_gap = np.mean(gap)
    
    return mean_test_score, mean_gap


## Multi-class

In [ ]:
study_name = "multiclassifier_optimization_norm"
storage_name = f"sqlite:///Optuna_Trainings/Big/{study_name}.db"
study = optuna.create_study(
    study_name=study_name,
    storage=storage_name,
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(seed=42),
    load_if_exists=True
)

obj = partial(objective_function, x=X_positive_norm, y=y_positive)
study.optimize(
    obj, 
    n_trials=100, 
    show_progress_bar=True, 
    gc_after_trial=True
)

In [ ]:
study_name = "multiclassifier_optimization_norm"
storage_name = f"sqlite:///Optuna_Trainings/Big/{study_name}.db"

study_name_2 = "multi_classifier_optimization_l2_norm.db"
storage_name_2 = f"sqlite:///Optuna_Trainings/Small/{study_name_2}.db"

scores, gaps = get_scores_gaps(
    [study_name, study_name_2],
    [storage_name, storage_name_2]
)

fig, ax = get_subplots()

ax.set_title("Test Score vs Train-Test Gap for best \n classification models", fontsize=fontsize, y=1.05)
for sc, gp, cl in zip(scores, gaps, ["blue", "red"]):
    ax.scatter(gp, sc, color=cl, alpha=0.9)
ax.legend(["New dataset", "Old dataset"])

ax.set_xlabel("Train-Test Gap", fontsize=fontsize)
ax.set_ylabel("Test Score", fontsize=fontsize)


In [ ]:
best= [trial for trial in study.best_trials if trial.values[0] > 0.2 and trial.values[1] < 0.1][0]

print(best.values)
print(best.params)


### Model Test

In [ ]:
x_train, x_test, y_train, y_test = sk.model_selection.train_test_split(X_positive_norm, y_positive, test_size=0.3, random_state=42)

In [ ]:
model_lr = sk.linear_model.LogisticRegression( C=0.1911276,  
            solver="saga",
            class_weight="balanced",
            max_iter=1000, 
            random_state=42
        )

model_rf = sk.ensemble.RandomForestClassifier(
            n_estimators=123,
            max_depth=1,
            class_weight="balanced",
            random_state=42
)

In [ ]:
np.random.seed(42)
random_labels = np.random.randint(0, 6, size=x_test.shape[0])

print(sk.metrics.classification_report(y_test, random_labels))
print(sk.metrics.confusion_matrix(y_test, random_labels))
print(sk.metrics.f1_score(y_test, random_labels, average="macro"))

#### Logistic Regression

In [ ]:
scores = []
Kfold = sk.model_selection.KFold(n_splits=5, random_state=42, shuffle=True)
for train, test in Kfold.split(X_positive_norm, y_positive):
    model_lr.fit(X_positive_norm[train], y_positive[train])
    y_pred = model_lr.predict(X_positive_norm[test])
    scores.append(sk.metrics.f1_score(y_positive[test], y_pred, average="macro"))

In [ ]:
get_model_stats(model_lr, X_positive_norm, y_positive, average="macro")

##### Feature Importance

In [ ]:
model_lr.fit(X_positive_norm, y_positive)
coefficients = model_lr.coef_
for i in range(6):
    near_zero = np.sum(np.abs(coefficients[i]) < 1e-4)
    # print(f"Class {i}: {near_zero} near-zero coefficients out of {coefficients.shape[1]}")
feature_importance = np.abs(coefficients).mean(axis=0)
top_k = 30
top_feature_indices = np.argsort(feature_importance)[-top_k:][::-1]

for rank, idx in enumerate(top_feature_indices, 1):
    # print(f"{rank:2d}. Feature {idx:3d}: {all_names_used[idx]:40s} | Importance: {feature_importance[idx]:.4f}")
    pass


In [ ]:
# ===== Visualization =====
fig, ax = get_subplots(figsize=(12, 6))
# Plot 1: Overall feature importance
# ax = axes[0]
top_features = [all_names_used[i] for i in top_feature_indices]
top_scores = feature_importance[top_feature_indices]

ax.barh(range(len(top_features)), top_scores, color='steelblue')
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features, fontsize=10)
ax.set_xlabel('Average |Weight| Across All Classes', fontsize=10)
ax.set_title('Top 30 Most Important Features - Logistic Regression', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)
# ax = axes[1]
top_20_idx = top_feature_indices[:20]
nto = np.array([ idx for idx in new_to_old.keys() ])
heatmap_data = coefficients[nto][:, top_20_idx].T  # Shape: (20 features, 6 classes)

fig, ax = get_subplots(figsize=(12, 10))
im = ax.imshow(heatmap_data, cmap='RdBu_r', aspect='auto', vmin=-1, vmax=1)
ax.set_yticks(range(20))
ax.set_yticklabels([all_names_used[i] for i in top_20_idx], fontsize=14)
ax.set_xticks(range(6))
ax.set_xticklabels([class_short_names[i] for i in range(6)], fontsize=14)
ax.set_title('Feature Weights per Class (Top 20 Features)', fontsize=14, fontweight='bold')
plot.colorbar(im, ax=ax, label='Weight')
ax.tick_params(axis='x', rotation=90)


plot.tight_layout()
plot.show()


#### Random Forest

In [ ]:
model_rf.fit(X_positive_norm, y_positive)
get_model_stats(model_rf, X_positive_norm, y_positive, average="macro")

##### Feature Importance

In [ ]:
explainer = shap.TreeExplainer(model_rf)
shap_values = explainer.shap_values(X_positive_norm)

importances = pd.DataFrame(
    np.abs(shap_values).mean(axis=0),
    index=all_names_used,
    columns=model_rf.classes_
)
importances["mean"] = importances.mean(axis=1)
importances.sort_values("mean", ascending=False, inplace=True)

nto = np.array([ idx for idx in new_to_old.keys() ])

importance_matrix = importances[importances["mean"] > 0.0][nto].values


In [ ]:
fig, ax = get_subplots(figsize=(12, 10))
curr_matrix = importance_matrix[:20]
cmap=ax.imshow(curr_matrix, cmap='RdBu_r', aspect='auto', )
cmap =fig.colorbar(cmap, ax=ax)
cmap.set_label("Mean Importance", fontsize=14)

# ax.set_xticks(np.arange(-0.5, curr_matrix.shape[1], 1), minor=True)
# ax.set_yticks(np.arange(-0.5, curr_matrix.shape[0], 1), minor=True)
# ax.grid(False,which="major")


ax.set_yticks(range(20))
ax.set_yticklabels(list(importances[importances["mean"] > 0.0][:20].index), fontsize=12)
ax.set_xticks(range(6))
ax.set_xticklabels([class_short_names[i] for i in range(6)], fontsize=14)
ax.set_title("Shap values for top 20 features\n Random Forest", fontsize=18, y=1.08)
ax.tick_params(axis='x', rotation=90)
plot.show()



#### MLP

In [ ]:
class MLP_Classifier(nn.Module):

    def __init__(self, input_dim, output_dim, hidden_dim=[128,64,32], dropout=0.3):
        super(MLP_Classifier, self).__init__()
        layers = []
        prev_dim = input_dim
        
        for h_dim in hidden_dim:
            layers.append(self._relu_layer(prev_dim, h_dim, dropout))
            prev_dim = h_dim
        
        layers.append(nn.Linear(prev_dim, output_dim))
        self.layers = nn.ModuleList(layers)

    @staticmethod
    def _relu_layer(prev_dim, h_dim, dropout):
        layer = nn.Sequential(
            nn.Linear(prev_dim, h_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        return layer

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x



In [ ]:
def train_model(x_t, y_t, x_v, y_v, output_dim, hidden_dim, dropout=0.3, lr=1e-3, batch_size=32, epochs=100, weight_decay=0.1,  SAVE_MODEL=False):
    input_dim = x_t.shape[1]
    model = MLP_Classifier(input_dim, output_dim, hidden_dim, dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, factor=0.5, patience=5)

    train_dataset = torch.utils.data.TensorDataset(x_t, y_t)
    len_train_set = len(train_dataset)
    val_dataset = torch.utils.data.TensorDataset(x_v, y_v)
    len_val_set = len(val_dataset)

    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size)

    best_val_loss = float('inf')
    patience_counter = 0
    patience_limit = 10
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        for x_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(x_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x_batch.size(0)

        train_loss /= len_train_set
        train_losses.append(train_loss)

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                outputs = model(x_batch)
                loss = criterion(outputs, y_batch)
                val_loss += loss.item() * x_batch.size(0)

        val_loss /= len_val_set
        val_losses.append(val_loss)

        # print(f"Epoch {epoch+1}/{epochs} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f}")
        scheduler.step(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            # best_model  = model.state_dict()
            best_model_state = copy.deepcopy(model.state_dict())
            if SAVE_MODEL:
                torch.save(model.state_dict(), "best_mlp_model.pth")
        else:
            patience_counter += 1
            if patience_counter >= patience_limit:
                print("Early stopping triggered.")
                break

    best_model = MLP_Classifier(input_dim, output_dim, hidden_dim, dropout)
    best_model.load_state_dict(best_model_state)
    best_model.eval()
    return best_model, train_losses, val_losses, model

def optimization(trial, x_t, y_t):
    num_hidden = trial.suggest_int("num_hidden", 1, 3)
    hidden = []
    for i in range(num_hidden):
        hidden.append(trial.suggest_int(f"hidden_{i}", 16, 64))

    dropout = trial.suggest_float("dropout", 0.1, 0.5)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [16, 32, 64])

    weight_decay = trial.suggest_float("weight_decay", 1e-5, 1e-1, log=True)

    Kfold = sk.model_selection.StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
    f1_val_scores = []
    gap_scores = []

    for train_idx, val_idx in Kfold.split(x_t, y_t):
        x_train_fold = x_t[train_idx]
        y_train_fold = y_t[train_idx]
        x_val_fold = x_t[val_idx]
        y_val_fold = y_t[val_idx]
        model, train_losses, val_losses, _ = train_model(x_t=x_train_fold, y_t=y_train_fold, x_v=x_val_fold, y_v=y_val_fold, output_dim=6, hidden_dim=hidden, dropout=dropout, lr=lr, batch_size=batch_size, weight_decay=weight_decay)

        model.eval()
        with torch.no_grad():
            te_outputs = model(x_val_fold)
            te_loss = nn.CrossEntropyLoss()(te_outputs, y_val_fold).item()
            te_preds = torch.argmax(te_outputs, dim=1)
            f1_val = sk.metrics.f1_score(y_val_fold, te_preds, average="macro")
            f1_val_scores.append(f1_val)

            tr_outputs = model(x_train_fold)
            tr_preds = torch.argmax(tr_outputs, dim=1)
            f1_tr = sk.metrics.f1_score(y_train_fold, tr_preds, average="macro")
            gap_scores.append(f1_tr - f1_val)

        
    
    return np.mean(f1_val_scores), np.mean(gap_scores)


In [ ]:
x_tr, x_te, y_tr, y_te = sk.model_selection.train_test_split(X_positive, y_positive, test_size=0.2, random_state=42)

scaler = sk.preprocessing.StandardScaler()
x_tr = scaler.fit_transform(x_tr)
x_te = scaler.transform(x_te)

# x_tr_sm, x_va, y_tr_sm, y_va = sk.model_selection.train_test_split(x_tr, y_tr, test_size=0.3, random_state=42)


x_tr = torch.tensor(x_tr, dtype=torch.float32)
y_tr = torch.tensor(y_tr, dtype=torch.long)

x_te = torch.tensor(x_te, dtype=torch.float32)
y_te = torch.tensor(y_te, dtype=torch.long)




In [ ]:
study_name = "MLP_optimization"
storage_name = f"sqlite:///Optuna_Trainings/Big/{study_name}.db"
study = optuna.create_study(
    study_name=study_name,
    storage=storage_name,
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(seed=42),
    load_if_exists=True
)

obj = partial(optimization, x_t=x_tr, y_t=y_tr)
study.optimize(
    obj, 
    n_trials=100, 
    show_progress_bar=True, 
    gc_after_trial=True
)


In [ ]:
study_name = "MLP_optimization"
storage_name = f"sqlite:///Optuna_Trainings/Big/{study_name}.db"
scores, gaps = get_scores_gaps(
    [study_name],
    [storage_name])

fig, ax = get_subplots()

ax.scatter(gaps[0], scores[0], color="blue", alpha=0.9)
plot.show()



In [ ]:
study = optuna.load_study(study_name=study_name, storage=storage_name)
best= [trial for trial in study.best_trials if trial.values[0] > 0.16 and 0<trial.values[1] ][0]
print(best.values)
print(best.params)



### Event Horizon

In [ ]:
suffix = ["_2W", "", "_5D", "_3D", "_1D"]
days = [14, 7, 5, 3, 1]
names = ["X_positive", "X_total_positive", "y_positive"]

f1_macros = []
tfdif = sk.feature_extraction.text.TfidfTransformer(norm="l2", use_idf=True, smooth_idf=True)

for suf, da in zip(suffix, days):
    X_p = names[0] + suf + ".npy"
    X_t_p = names[1] + suf + ".npy"
    y_p = names[2] + suf + ".npy"

    X_p = np.load(os.path.join(EVENT_DIR, X_p))
    X_t_p = np.load(os.path.join(EVENT_DIR, X_t_p))
    y_p = np.load(os.path.join(EVENT_DIR, y_p))
    tfdif.fit(X_t_p)

    X_p_n = tfdif.transform(X_p).toarray()

    get_model_stats(model_lr, X_p_n, y_p, average="macro")

    x_train, x_test, y_train, y_test = sk.model_selection.train_test_split(X_p_n, y_p, test_size=0.3, random_state=42)
    model_lr.fit(x_train, y_train)
    preds = model_lr.predict(x_test)
    f1_macros.append(sk.metrics.f1_score(y_test, preds, average="macro"))

In [ ]:
fig, ax = get_subplots()

ax.scatter(days, f1_macros)
ax.set_title("Evolution of f1 macro with event horizon", fontsize=18)
ax.set_xlabel("Event horizon (Days)", fontsize=18)
ax.set_ylabel("F1 metric", fontsize=18)
ax.set_ylim(0,0.3)

plot.show()


## Binary Classifier

### Best Model

In [ ]:
study_name = "logs_optimization_norm_binary"
storage_name = f"sqlite:///Optuna_Trainings/Big/{study_name}.db"
study = optuna.create_study(
    study_name=study_name,
    storage=storage_name,
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(seed=42),
    load_if_exists=True
)

obj = partial(objective_function, x=X_full_norm, y=y_binary, average="binary")
study.optimize(
    obj, 
    n_trials=100, 
    show_progress_bar=True, 
    gc_after_trial=True
)

In [ ]:
study_name = "logs_optimization_norm_binary"
storage_name = f"sqlite:///Optuna_Trainings/Big/{study_name}.db"
scores, gaps = get_scores_gaps(
    [study_name],
    [storage_name])

fig, ax = get_subplots()

ax.scatter(gaps[0], scores[0], color="blue", alpha=0.9)
plot.show()


In [ ]:
best= [trial for trial in study.best_trials if trial.values[0] > 0.2 and 0<trial.values[1] ][0]
print(best.values)
print(best.params)

#### Random Forest

In [ ]:
model = sk.ensemble.RandomForestClassifier(
    n_estimators = 100,
    max_depth = 1,
    class_weight = "balanced",
    random_state=42
)
model, test, gap = model_train(model, X_full_norm, y_binary, "binary")

print(test, np.mean(test))
print(gap, np.mean(gap))

indexes = np.arange(X_full_norm.shape[0])
train, test = sk.model_selection.train_test_split(indexes, test_size=0.3, random_state=42)

model.fit(X_full_norm[train], y_binary[train])
y_pred = model.predict(X_full_norm[test])
print(sk.metrics.classification_report(y_binary[test], y_pred))
print(sk.metrics.confusion_matrix(y_binary[test], y_pred))
print(sk.metrics.f1_score(y_binary[test], y_pred, average="binary"))

model.fit(X_full_norm, y_binary)


##### Feature Importance

In [ ]:

records = []
for tree in model.estimators_:
    t = tree.tree_
    n_total = t.n_node_samples[0]  # root node sample count

    for node_id in range(t.node_count):

        feature_index = t.feature[node_id]
        left  = t.children_left[node_id]
        right = t.children_right[node_id]

        # Weighted impurity decrease = proxy for this split's importance
        importance = (
            (t.n_node_samples[node_id] / n_total) * t.impurity[node_id]
            - (t.n_node_samples[left]    / n_total) * t.impurity[left]
            - (t.n_node_samples[right]   / n_total) * t.impurity[right]
        )

        # Dominant class on each side of the split
        left_class  = model.classes_[np.argmax(t.value[left][0])]
        right_class = model.classes_[np.argmax(t.value[right][0])]

        records.append({
            'threshold':   t.threshold[node_id],
            'feature': all_names_used[feature_index],
            'importance':  importance,
            'left_class':  left_class,
            'right_class': right_class,
        })

df = pd.DataFrame(records)
df.groupby(["feature", "left_class", "right_class"]).agg({"threshold":list})

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_full_norm)
shap_values = shap_values[:,:,1]  

importances = pd.DataFrame(
    np.abs(shap_values).mean(axis=0),
    index=all_names_used,
    columns=["Positive"]
)
importances["mean"] = importances.mean(axis=1)
importances.sort_values("mean", ascending=False, inplace=True)
importance_matrix = importances[importances["mean"] > 0.0].values

In [ ]:

fig, ax = get_subplots(figsize=(10, 8))
# Get only the first column (Positive Class), reshape to keep 2D for imshow
curr_matrix = importance_matrix[:20, [0]]  # Shape: (20, 1)

cmap_obj = ax.imshow(curr_matrix, cmap="viridis", aspect="auto")
cbar = fig.colorbar(cmap_obj, ax=ax)
cbar.set_label("Mean Importance", fontsize=14)

# Grid lines for single column
ax.set_xticks(np.arange(-0.5, 1, 1), minor=True)
ax.set_yticks(np.arange(-0.5, 20, 1), minor=True)
ax.grid(which="minor", color="gray", linestyle='-', linewidth=0.5)
ax.tick_params(which="minor", size=0)  # Hide minor tick marks
ax.grid(False, which="major")

# Set axis labels
ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
ax.yaxis.set_major_locator(ticker.MultipleLocator(1))
ax.set_xticks([0])
ax.set_xticklabels(["Positive"], fontsize=14)
ax.set_yticklabels([""] + list(importances[importances["Positive"] > 0.0].index[:20]), fontsize=12)

ax.set_title("SHAP values for top 20 features\n Binary Classifier (Fault Prediction)", fontsize=18, y=1.08)
plot.tight_layout()
plot.show()


#### Logistic Regression

In [ ]:
model = sk.linear_model.LogisticRegression(
    C = 0.13293,
    class_weight="balanced",
    solver="lbfgs",
    random_state=42
)
model, test, gap = model_train(model, X_full_norm, y_binary, "binary")

print(test, np.mean(test))
print(gap, np.mean(gap))

indexes = np.arange(X_full_norm.shape[0])
train, test = sk.model_selection.train_test_split(indexes, test_size=0.3, random_state=42)

model.fit(X_full_norm[train], y_binary[train])
y_pred = model.predict(X_full_norm[test])
print(sk.metrics.classification_report(y_binary[test], y_pred))
print(sk.metrics.confusion_matrix(y_binary[test], y_pred))
print(sk.metrics.f1_score(y_binary[test], y_pred, average="binary"))

model.fit(X_full_norm, y_binary)


##### Feature Importance

In [ ]:
coefficients = model.coef_
feature_importance = np.abs(coefficients).mean(axis=0)
top_k = 30
top_feature_indices = np.argsort(feature_importance)[-top_k:][::-1]

top_features = [all_names_used[i] for i in top_feature_indices]
top_scores = feature_importance[top_feature_indices]

top_20_idx = top_feature_indices[:20]
heatmap_data = coefficients[:, top_20_idx].T  # Shape: (20 features, 2 classes)

In [ ]:
fig, ax = get_subplots(figsize=(12, 10))
im = ax.imshow(heatmap_data, cmap='RdBu_r', aspect='auto')
ax.set_yticks(range(20))
ax.set_yticklabels([all_names_used[i] for i in top_20_idx], fontsize=14)
ax.set_xticks(range(1))
ax.set_xticklabels(["Positive Class"], fontsize=14)
ax.set_title('Feature Weights per Class (Top 20 Features)', fontsize=14, fontweight='bold')
plot.colorbar(im, ax=ax, label='Weight')
ax.grid(False, which="both", axis="x" )
ax.set_yticks(np.arange(-0.5, 20, 1), minor=True)
ax.grid(which="minor", axis="y", color="gray", linestyle='-', linewidth=0.5)
ax.grid(False, which="major", axis="y")

ax.tick_params(axis='x', rotation=0)


plot.tight_layout()
plot.show()


### PU Learning

In [ ]:
import numpy as np
import sklearn as sk
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score, precision_score, recall_score
from collections import Counter

class PULearner:
    """Positive-Unlabeled Learning using Spy Technique"""
    
    def __init__(self, base_classifier, spy_ratio=0.15, reliable_threshold=0.1):
        self.base_classifier = base_classifier
        
        self.spy_ratio = spy_ratio
        self.reliable_threshold = reliable_threshold
        self.threshold_ = None
        self.final_classifier_ = None
    
    def spy_technique(self, X_pos, X_un):
        """Use spy technique to find threshold for reliable negatives."""
        n_spy = int(len(X_pos) * self.spy_ratio)
        np.random.seed(42)
        spy_indices = np.random.choice(len(X_pos), size=n_spy, replace=False)
        
        spy_mask = np.zeros(len(X_pos), dtype=bool)
        spy_mask[spy_indices] = True
        
        X_spy = X_pos[spy_mask]
        X_pos_remaining = X_pos[~spy_mask]
        X_un_with_spy = np.vstack([X_un, X_spy])
        
        X_train = np.vstack([X_pos_remaining, X_un_with_spy])
        y_train = np.array([1] * len(X_pos_remaining) + [0] * len(X_un_with_spy))
        
        self.base_classifier.fit(X_train, y_train)
        spy_probs = self.base_classifier.predict_proba(X_spy)[:, 0]
        self.threshold_ = np.percentile(spy_probs, 100 * (1 - self.reliable_threshold))
        
        return self.threshold_
    
    def fit(self, X_pos, X_un, return_indices=False):
        """Fit PU learner and identify reliable negatives."""
        print(f"Step 1: Spy technique with {self.spy_ratio*100:.0f}% spies...")
        self.spy_technique(X_pos, X_un)
        print(f"  Threshold set at: {self.threshold_:.4f}")
        
        print(f"\nStep 2: Identifying reliable negatives...")
        un_probs = self.base_classifier.predict_proba(X_un)[:, 0]
        reliable_negative_mask = un_probs >= self.threshold_
        X_reliable_negative = X_un[reliable_negative_mask]
        
        print(f"  Found {len(X_reliable_negative)} reliable negatives out of {len(X_un)} un")
        print(f"  Ratio: {len(X_reliable_negative)/len(X_un)*100:.1f}%")
        
        print(f"\nStep 3: Training final classifier...")
        X_final = np.vstack([X_pos, X_reliable_negative])
        y_final = np.array([1] * len(X_pos) + [0] * len(X_reliable_negative))
        
        self.final_classifier_ = sk.base.clone(self.base_classifier)
        self.final_classifier_.fit(X_final, y_final)
        print(f"  Training complete. Class distribution: {Counter(y_final)}")
        
        if return_indices:
            reliable_indices = np.where(reliable_negative_mask)[0]
            return X_reliable_negative, reliable_indices, un_probs
        
        return X_reliable_negative, un_probs
    
    def predict(self, X):
        return self.final_classifier_.predict(X)
    
    def predict_proba(self, X):
        return self.final_classifier_.predict_proba(X)


In [ ]:
model = sk.linear_model.LogisticRegression(
            C=0.13293,
            solver="lbfgs",
            class_weight="balanced",
            max_iter=1000,
            random_state=42
        )
pu_learner = PULearner(
    base_classifier=model,
    spy_ratio=0.15,
    reliable_threshold=0.1
)

un_mask = y_binary == 0
X_un  = X_full_norm[un_mask]
X_pos = X_full_norm[~un_mask]
X_reliable_neg, reliable_neg_indices, neg_probs = pu_learner.fit(
    X_pos, 
    X_un, 
    return_indices=True
)

print(f"Reliable negatives: {len(X_reliable_neg)}")

X_reliable_neg = X_full[un_mask][reliable_neg_indices]
X_reliable_neg_norm = X_full_norm[un_mask][reliable_neg_indices]
positive_counts = X_positive.copy()

X_reliable_full = np.vstack([X_positive, X_reliable_neg])
X_reliable_full_norm = np.vstack([X_full_norm[~un_mask], X_full_norm[un_mask][reliable_neg_indices]])
y_reliable = np.array([1] * len(X_positive) + [0] * len(X_reliable_neg))


In [ ]:
study_name = "logs_optimization_norm_binary_double_reliable"
storage_name = f"sqlite:///Optuna_Trainings/Big/{study_name}.db"
study = optuna.create_study(
    study_name=study_name,
    storage=storage_name,
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(seed=42),
    load_if_exists=True
)

obj = partial(objective_function, x=X_reliable_full_norm, y=y_reliable, average="binary")
study.optimize(
    obj, 
    n_trials=100, 
    show_progress_bar=True, 
    gc_after_trial=True
)


In [ ]:
study_name = "logs_optimization_norm_binary_double_reliable"
storage_name = f"sqlite:///Optuna_Trainings/Big/{study_name}.db"
scores, gaps = get_scores_gaps(
    [study_name],
    [storage_name])

fig, ax = get_subplots()

ax.scatter(gaps[0], scores[0], color="blue", alpha=0.9)
plot.show()


In [ ]:
best= [trial for trial in study.best_trials if trial.values[0] > 0.2 and 0<trial.values[1] ][0]
print(best.values)
print(best.params)


#### Logistic Regression

In [ ]:
model_reliable_lr = sk.linear_model.LogisticRegression(
    C = 0.66471,
    class_weight="balanced",
    random_state=42
)


get_model_stats(model_reliable_lr, X_reliable_full, y_reliable, average="binary")
model_reliable_lr.fit(X_reliable_full, y_reliable)


##### Feature Importance

In [ ]:
coefficients = model_reliable_lr.coef_
feature_importance = np.abs(coefficients).mean(axis=0)
top_k = 30
top_feature_indices = np.argsort(feature_importance)[-top_k:][::-1]

top_features = [all_names_used[i] for i in top_feature_indices]
top_scores = feature_importance[top_feature_indices]

top_20_idx = top_feature_indices[:20]
heatmap_data = coefficients[:, top_20_idx].T  # Shape: (20 features, 2 classes)

In [ ]:
fig, ax = get_subplots(figsize=(12, 10))
im = ax.imshow(heatmap_data, cmap='viridis', aspect='auto', vmin=-1, vmax=9)
ax.set_yticks(range(20))
ax.set_yticklabels([all_names_used[i] for i in top_20_idx], fontsize=14)
ax.set_xticks(range(1))
ax.set_xticklabels(["Positive Class"], fontsize=14)
ax.set_title('Feature Weights per Class (Top 20 Features)', fontsize=14, fontweight='bold')
plot.colorbar(im, ax=ax, label='Weight')
ax.grid(False, which="both", axis="x" )
ax.set_yticks(np.arange(-0.5, 20, 1), minor=True)
ax.grid(which="minor", axis="y", color="gray", linestyle='-', linewidth=0.5)
ax.grid(False, which="major", axis="y")

ax.tick_params(axis='x', rotation=0)


plot.tight_layout()
plot.show()


###### Separate between Positive / Negative classification

In [ ]:
positive_coefficients = np.where(coefficients[0] > 0)[0]
negative_coefficients = np.where(coefficients[0] < 0)[0]

positive_coefficients = sorted(positive_coefficients, key = lambda x: coefficients[0][x], reverse=True)[:20]
negative_coefficients = sorted(negative_coefficients, key = lambda x: np.abs(coefficients[0][x]), reverse=True)[:20]

heatmap_positive = coefficients[:,positive_coefficients].T
heatmap_negative = coefficients[:,negative_coefficients].T

In [ ]:
fig, ax = get_subplots(figsize=(12, 10))
im = ax.imshow(heatmap_positive, cmap='viridis', aspect='auto', vmin=0, vmax=9)
ax.set_yticks(range(20))
ax.set_yticklabels([all_names_used[i] for i in positive_coefficients], fontsize=14)
ax.set_xticks(range(1))
ax.set_xticklabels(["Positive Class"], fontsize=14)

ax.set_title('Feature Weights Positive Class (Top 20 Features)', fontsize=14, fontweight='bold')
plot.colorbar(im, ax=ax, label='Weight')
ax.grid(False, which="both", axis="x" )
ax.set_yticks(np.arange(-0.5, 20, 1), minor=True)
ax.grid(which="minor", axis="y", color="gray", linestyle='-', linewidth=0.5)
ax.grid(False, which="major", axis="y")

fig, ax = get_subplots(figsize=(12, 10))
im = ax.imshow(np.abs(heatmap_negative), cmap='viridis', aspect='auto', vmin=0, vmax=3)
ax.set_yticks(range(20))
ax.set_yticklabels([all_names_used[i] for i in negative_coefficients], fontsize=14)
ax.set_xticks(range(1))
ax.set_xticklabels(["Negative Class"], fontsize=14)

ax.set_title('Feature Weights Negative Class (Top 20 Features)', fontsize=14, fontweight='bold')
plot.colorbar(im, ax=ax, label='Absolute Weight')
ax.grid(False, which="both", axis="x" )
ax.set_yticks(np.arange(-0.5, 20, 1), minor=True)
ax.grid(which="minor", axis="y", color="gray", linestyle='-', linewidth=0.5)
ax.grid(False, which="major", axis="y")

plot.show()


#### Random Forest

In [ ]:
model_reliable_rf = sk.ensemble.RandomForestClassifier(
            n_estimators=120,
            max_depth=1,
            random_state=42,
            class_weight="balanced"
        )
        
get_model_stats(model_reliable_rf, X_reliable_full, y_reliable, average="binary")
model_reliable_rf.fit(X_reliable_full, y_reliable)

### Live mode

In [ ]:
model_used = model_reliable_lr

In [ ]:
def get_extension(events, charger):
    specific_events = events[events["device_id"] == charger]
    min_, max_ = sicharge_family[sicharge_family["charging_station_id"] == charger][["min", "max"]].values[0]
    if pd.isnull(min_) or pd.isnull(max_):
        return [], []
    min_ = pd.to_datetime(min_).tz_localize("UTC").normalize()
    max_ = pd.to_datetime(max_).tz_localize("UTC").normalize() + pd.Timedelta(days=1)
    dates = pd.date_range(min_, max_, freq="1D")

    weekly_counts = []

    extension = []
    temp = [[dates[0]], -1]
    for date in dates:
        weekly_counts = get_event_count_no_tot(specific_events, date, pd.Timedelta("1W"), threshold_event=0, cols=cols)
        x_cont = np.zeros(len(all_names_used))
        for name, cnt in weekly_counts:
            idx = name_idx[name]
            x_cont[idx] = cnt
        y_pred = model_used.predict(x_cont.reshape(1, -1))[0]
        if temp[1] == -1:
            temp[1] = y_pred
        elif y_pred != temp[1]:
            temp[0].append(date)
            extension.append(temp)
            temp = [[date], y_pred]
    
    temp[0].append(dates[-1])
    extension.append(temp)
    return extension, dates

def process_bucket_extension(bucket_nmr):
    chargers_in_bucket = charger_locations[charger_locations["bucket"] == bucket_nmr]["device_id"].tolist()
    events = pd.read_parquet(os.path.join(EVENT_DIR, "Buckets", f"bucket_{bucket_nmr}.parquet"))
    events = preprocess_events_counts(events)
    events["event_time"] = (events["event_time"]).dt.tz_localize("UTC")
    results = Parallel(n_jobs=-1, verbose=10)(
        delayed(get_extension)(events, charger)
        for charger in chargers_in_bucket
    )
    extensions = {charger: results[i][0] for i, charger in enumerate(chargers_in_bucket)}

    return extensions


#### Specific Charger

In [ ]:
specific_charger = "8qgG8Z"
bucket_charger = charger_locations[charger_locations["device_id"] == specific_charger]["bucket"].values[0]
events = pd.read_parquet(os.path.join(EVENT_DIR, "Buckets", f"bucket_{bucket_charger}.parquet"))
events = preprocess_events_counts(events)
events["event_time"] = (events["event_time"]).dt.tz_localize("UTC")

In [ ]:
specific_charger = "8qgG8Z"
specific_events = events[events["device_id"] == specific_charger]
extension,dates = get_extension(specific_events, specific_charger)

specific_tickets = charger_id
specific_tickets = [el for el in specific_tickets if el[0] == specific_charger]
specific_dates = [pd.Timestamp(str(el[1])) for el in specific_tickets]

event_histograms = {
    name : np.histogram(val["event_time"], bins=dates)[0]
    for name, val in specific_events.groupby("name")
}

##### Visualization

In [ ]:
fig = plot.figure(figsize=(12, 6))
gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3)
colors = color_dic["default"]
ax_top = fig.add_subplot(gs[0, :])
ax_b_r = fig.add_subplot(gs[1, 1])
ax_b_l = fig.add_subplot(gs[1, 0], sharey=ax_b_r)
axs = [ax_top, ax_b_l, ax_b_r]
axs_twinx = [a.twinx() for a in axs]
prepare_subplots(axs, GRID=1)
prepare_subplots(axs_twinx, GRID=1)

good = [el[0] for el in extension if el[1] == 0]
bad  = [el[0] for el in extension if el[1] == 1]


aou = pd.Series(event_histograms["allOutletsUnavailable"], index=dates[:-1])
pc = pd.Series(event_histograms["PowerConverters"], index=dates[:-1])
dc_codes = pd.Series(event_histograms["Outlet_DCX_codes"], index=dates[:-1])
aou_rolling = aou.rolling(window=7, min_periods=7).sum()
pc_rolling = pc.rolling(window=7, min_periods=7).sum()
dc_codes_rolling = dc_codes.rolling(window=7, min_periods=7).sum()

for a, a_t in zip(axs, axs_twinx):

    for interval in extension:
        color = "red" if interval[1] == 1 else "green"
        a.fill_betweenx([0,1], interval[0][0], interval[0][-1], color=color, alpha=0.3, label="Predicted Normal Operation" if color=="green" else "Predicted Fault")


    a.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    a.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    # a.xaxis.set_minor_locator(mdates.WeekdayLocator(byweekday=mdates.MO))
    a.xaxis.set_minor_locator(mdates.MonthLocator(interval=1))
    a.scatter(specific_dates, [0.5]*len(specific_dates), color="blue", label="Actual Events", marker="x")

    a_t.plot(dates[:-1], aou_rolling.values, label="All Outlets Unavailable ", linestyle=(0, (3,2)), color=colors[0])
    a_t.plot(dates[:-1], pc_rolling.values, label="Power Converters ", linestyle=(0, (3,2)), color=colors[5])
    a_t.plot(dates[:-1], dc_codes_rolling.values, label="Outlet DCX Codes ", linestyle=(0, (3,2)), color=colors[6])

    a.set_yticks([])
    a.set_xlim(dates[0], dates[-1])
    a.set_ylim(0,1)

for a_t in [axs_twinx[0], axs_twinx[2]]:
    a_t.set_ylabel("7-Day Rolling Sum of Events", fontsize=12)
axs[1].set_xlim()
for a in axs[1:]:
    a.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    # a.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO))

    a.xaxis.set_major_locator(mdates.DayLocator(interval=10))
    a.xaxis.set_minor_locator(mdates.DayLocator(interval=2))
    a.tick_params(axis='x', rotation=45)


axs[1].xaxis.set_major_locator(mdates.DayLocator(interval=20))
axs[1].xaxis.set_minor_locator(mdates.DayLocator(interval=5))

legend = [
    mpatches.Patch(color="green", alpha=0.3, label="Predicted Normal Operation"),
    mpatches.Patch(color="red", alpha=0.3, label="Predicted Fault"),
    mlines.Line2D([], [], color="blue", marker="x", linestyle='None', label="Ticket Submission")
]
legend1 = fig.legend(handles=legend,
                       loc='upper right',
                       bbox_to_anchor=(0.12, 0.9))

legend = [
    mlines.Line2D([], [], color=colors[0], linestyle=(0, (3,2)), label="All Outlets Unavailable"),
    mlines.Line2D([], [], color=colors[5], linestyle=(0, (3,2)), label="Power Converters"),
    mlines.Line2D([], [], color=colors[6], linestyle=(0, (3,2)), label="Outlet DCX Codes")
]
legend2 = fig.legend(handles=legend, title="Event Type (7-Day rolling sum)",
                       loc='upper right',
                       bbox_to_anchor=(0.12, 0.75))

# axs[1].set_xlim(pd.Timestamp("2024-10-01", tz="UTC"), pd.Timestamp("2024-11-30", tz="UTC"))
# axs[2].set_xlim(pd.Timestamp("2025-12-01", tz="UTC"), pd.Timestamp("2026-03-10", tz="UTC"))

axs[1].set_xlim(pd.Timestamp("2024-11-11", tz="UTC"), pd.Timestamp("2025-03-30", tz="UTC"))
axs[2].set_xlim(pd.Timestamp("2026-01-01", tz="UTC"), pd.Timestamp("2026-02-18", tz="UTC"))

fig.suptitle(f"Predicted Fault Periods for Charger {specific_charger} \n New model using Raw Count", fontsize=20, y=1.02)
fig.tight_layout()
plot.show()


#### Overall Statistics

##### Dataset

###### Create

In [ ]:
all_extensions = {}
for i in range(10):
    extensions = process_bucket_extension(i)
    all_extensions.update(extensions)

In [ ]:
total_events = {}
for bucket_nmr in range(10):
    chargers_in_bucket = charger_locations[charger_locations["bucket"] == bucket_nmr]["device_id"].tolist()
    events = pd.read_parquet(os.path.join(EVENT_DIR, "Buckets", f"bucket_{bucket_nmr}.parquet"))
    events = preprocess_events_counts(events)
    for charger in tqdm(chargers_in_bucket):
        total_events[charger] = events[events["device_id"] == charger].shape[0]

In [ ]:
output_file = os.path.join(EVENT_DIR, "live_view_extensions.pkl")
# with open(output_file, 'wb') as f:
    # pickle.dump(all_extensions, f)
with open(os.path.join(EVENT_DIR, "live_view_extensions.pkl"), 'rb') as f:
    all_extensions = pickle.load(f)

###### Load

In [ ]:

with open(os.path.join(EVENT_DIR, "live_view_extensions.pkl"), 'rb') as f:
    all_extensions = pickle.load(f)

##### Preprocessing

In [ ]:

predicted_bad_per = {}
total_times = {}

for charger, extension in all_extensions.items():
    bad = [el[0][-1]-el[0][0] for el in extension if el[1]]
    min_, max_ = sicharge_family[sicharge_family["charging_station_id"] == charger][["min", "max"]].values[0]
    if pd.isnull(min_) or pd.isnull(max_):
        continue
    min_ = pd.to_datetime(min_).tz_localize("UTC").normalize()
    max_ = pd.to_datetime(max_).tz_localize("UTC").normalize() + pd.Timedelta(days=1)
    bad = sum(bad, start = pd.Timedelta('0s'))
    total_time = max_ - min_

    predicted_bad_per[charger] = (bad/total_time)

    total_times[charger] = total_time
    
data = np.array([(predicted_bad_per[charger], total_events[charger]) for charger in predicted_bad_per.keys()])

In [ ]:
len(list(set(charger_id[:,0])))
chargers_with_tickets = list(set(charger_id[:,0]))

In [ ]:
data_no_tickets = np.array([(predicted_bad_per[charger], total_events[charger]) for charger in predicted_bad_per.keys() if not charger in chargers_with_tickets])
data_tickets = np.array([(predicted_bad_per[charger], total_events[charger]) for charger in predicted_bad_per.keys() if charger in chargers_with_tickets])

predicted_bad_per_no_tickets = [predicted_bad_per[charger] for charger in predicted_bad_per.keys() if not charger in chargers_with_tickets ]
predicted_bad_per_tickets = [predicted_bad_per[charger] for charger in predicted_bad_per.keys() if charger in chargers_with_tickets ]

##### Visualization

In [ ]:
fig, ax  = get_subplots()
fig2, ax2  = get_subplots()
ax.hist([predicted_bad_per_no_tickets, predicted_bad_per_tickets], bins = np.arange(0,1,0.02), stacked=True, color=["g", "r"])

# ax2.scatter(data_no_tickets[:,0], data_no_tickets[:,1])
ax2.scatter(data_tickets[:,0], data_tickets[:,1], color="r", marker=".",alpha=0.3)
ax2.scatter(data_no_tickets[:,0], data_no_tickets[:,1], color="g", marker=".", alpha=0.3)

patches = [mpatches.Patch(facecolor="g", label="Without Tickets"),
           mpatches.Patch(facecolor="r", label="With Submitted Tickets")]

for a in [ax,ax2]:
    a.set_xlabel("Percentage of predicted abnormal", fontsize=18)
    a.legend(handles=patches)
ax.set_ylabel("# chargers", fontsize=18)
ax2.set_ylabel("# Events", fontsize=18)
ax2.set_yscale("log")
# ax2.set_ylim(-1e3,1e4)
scipy.

plot.show()
